- **Executive Summary:** Arxiv Forecasts
- **Team:** Roberto Albesiano, Maneesha Ampagouni, Bobby Zhang
- **Project Mentor:** Alec Traaseth
- **GitHub:** [arXiv Forecasts](https://github.com/albesianor/arxiv-heatmap)

## Abstract

The arXiv is the leading open-access repository for research preprints in fields such as Physics, Mathematics, and Computer Science. With a growing volume of daily submissions, the visibility and audience of individual preprints can vary significantly depending on posting dynamics. This project aims to develop a forecasting algorithm to predict arXiv posting behavior per category, with the following goal in mind:  
- **Short-term forecasting** for optimizing a preprint's *visibility*  

By analyzing historical arXiv posting data and usage statistics, we build models that can assist researchers in strategically choosing submission dates to increase their paper's exposure and readership.


## Stakeholders and KPIs

- **Researchers** are the primary stakeholders, aiming to increase the visibility and impact of their preprints on arXiv.
- The **short-term forecasting goal** is to help researchers choose days with fewer competing submissions in their category.
- The model is based on the **assumption** that preprint visibility is inversely related to the number of submissions on a given day — i.e., the fewer the competing papers, the higher the chance of being noticed. This is supported by analysis of arXiv usage data, which shows that usage remains fairly stable across weekdays and does not increase proportionally with the number of submissions, resulting in a modest but statistically significant positive correlation (≈ 0.25, p ≪ 0.01) between usage and total appearances.
- The main **KPI** is whether the model helps authors post on days with significantly fewer competing preprints than average.


## Data Collection and Cleaning

- **Data Source:** The dataset used in this project comes from the [arXiv Metadata Kaggle dataset](https://www.kaggle.com/datasets/Cornell-University/arxiv), which contains metadata for over **2.2 million** preprints submitted to arXiv since the late 1980s. Each entry includes fields such as `id`, `authors`, `title`, `categories`, `abstract`, `versions`, and more. For preprocessing, only the essential columns like `id`, `versions`, and `categories` are retained, reducing the dataset to around 2.2 million rows and 3–4 columns. The submission date is standardized using the first version date (`v1`) extracted from the `versions` field.

- **Pre-cleaning:** Removed abstract content to reduce dataset size, and retained only essential columns (`id`, `versions`, `categories`) for further processing.

- **Date Extraction:** Used the date of the first version (`v1`) of each preprint as the canonical submission date, due to inconsistencies in the `update_date` field.

- **v1 Date Pipeline:** Created a `date_extractor` function, applied it to the `versions` column, and saved the updated dataset with a new `date` column to `data/arxiv-id-date-categories.parquet`.

- **Category Formatting:** Split space-separated category strings into lists to standardize formatting for analysis.

- **Handling Deprecated Categories:** Identified and mapped outdated or deprecated category labels to current ones using a predefined dictionary.

- **Final Cleaning Step:** Replaced old categories with valid ones (including meta-categories where applicable), removed duplicates, and saved the cleaned dataset to `data/arxiv-metadata-cleaned.parquet`.


## Usage Data Summary

- **Overview**: Usage statistics provide a measure of user activity on arXiv, based on the number of daily connections to the site. This serves as a proxy for audience size and helps identify patterns in engagement that could influence the timing of preprint submissions.

- **Dataset:** We analyzed `arxiv-usage.parquet`, which logs daily connection counts to the arXiv website from 2024-01-01 to 2025-04-10.

- **Weekly Patterns:** Usage drops significantly on weekends. Mondays and Tuesdays typically show the highest connection counts.

- **Daily Variability:** Even among weekdays, usage shows notable fluctuations, hinting at influences beyond just the day of the week.

- **Hypothesis Testing:** An F-test comparing a baseline model (intercept-only) to a model with one-hot encoded weekdays yielded:
  - **F-test p-value:** 0.706

- **Conclusion:** The test fails to reject the null hypothesis — weekday does not significantly affect usage. The model “average + noise” suffices.

- **Correlation with Submissions**: A Pearson correlation analysis between the number of connections and the total number of article appearances per day showed a correlation coefficient of approximately **0.248** with a **p-value of 5.7e-08**. This indicates a weak but statistically significant positive relationship, suggesting that on busier days (in terms of submissions), usage tends to be slightly higher, but not strongly enough to discount our core visibility assumptions.

- **Implication:** These results guide short-term forecasting by clarifying when user activity (and thus potential visibility) peaks.


## Baseline Models

To evaluate short-term forecasting performance, we implemented and compared several simple baseline models. These models were trained on 150-day rolling windows and evaluated using 5-fold cross-validation, with a 5-business-day gap between training and validation splits, and a 15-business-day test horizon.

### Models Compared
- **Dummy Model**: Predicts the mean of the previous 150 days.
- **Time Regression (`t_reg`)**: Linear regression on time only (captures trend).
- **Day-of-Week Regression (`day_reg`)**: Linear regression on one-hot encoded day-of-week features (captures weekly seasonality).
- **Time + Day-of-Week Regression (`tday_reg`)**: Combines both trend and weekday information in a single linear model.

### Results
- **Best performing model**: `day_reg`, which performed best in 53.5% of the categories.
- **Mean improvement** over dummy model: **0.129**.
- **Maximum improvement** over dummy: **0.653**.

## Model Extensions (Advanced Models Under Exploration)

Beyond the baseline comparisons, we also began exploring more sophisticated forecasting models:
- **Holt-Winters (Triple Exponential Smoothing)**:
  - Both additive and multiplicative seasonalities were tested.
  - Tunable hyperparameters include trend, damped trend, and seasonality.
- **SARIMA / ARIMA**:
  - Captures autoregressive structure, moving averages, and seasonality in time series data.
- **Facebook Prophet**:
  - A flexible time series model that handles changepoints, holidays, and multiple seasonal patterns.

